In [58]:
import json 
import collections as coll
import pandas as pd
from pathlib import Path
import numpy as np
import tabulate

# Initialization

In [59]:
# 0 is most recent, 1 is prior.
def pull_json(num):
    path = list(Path(".").glob(f"{num}_*"))[0].__str__()
    with open(path, encoding = "utf-8") as f:
        current_json = json.load(f)
    return(current_json)

In [60]:
new_raw = pull_json(0)
old_raw = pull_json(1)

# Construction

## Find new

In [61]:
# Given JSON-as-dict, sorts the IDs.
def organize_new(x, file):
    x = file["bundles"][x]["strings"]
    x = {int(k):v for k, v in x.items()}
    x = pd.DataFrame(sorted(x.items()))
    x.columns = ["id", "name"]
    return(x)

In [62]:
# Get old and new, retain only new.
old = organize_new("MonsterName", old_raw)
new = organize_new("MonsterName", new_raw)
new_id_df = pd.merge(old, new, how = "outer", indicator = "right")
new_ids = new_id_df[new_id_df["right"] == "right_only"]["id"]

## Construct Daemon collections

In [63]:
# Given JSON-as-dict, sorts the IDs and keeps new ones.
def organize(x):
    x = new_raw["bundles"][x]["strings"]
    x = {int(k):v for k, v in x.items()}
    x = pd.DataFrame(sorted(x.items()))
    x.columns = ["id", "descr"]
    return(x)

In [64]:
# Acquire names.
name = organize("MonsterName")
name.columns = ["id", "name"]

# Acquire all main skills.
skill = organize("MonsterSkillDesc").rename(columns = {"descr": "skill"})

# Acquire all passive abilities.
ability = organize("MonsterAbilityDesc")
# Create daemon ID and ability ID columns, then pivot.
ability["a_id"] = ability["id"] % 100
ability["id"] = ability["id"] // 100
ability = ability.pivot(index = "id", columns = "a_id", values = "descr")
ability = ability.rename(columns = {1: "ability 1", 2: "ability 2"})

In [68]:
# Generate final new daemon database.
daemons = name.merge(skill.merge(ability, on = "id", how = "left"), on = "id")
daemons = daemons[daemons["id"].isin(new_ids)]

# Output

In [66]:
# NaN condition
def print_if(aspect, i):
    if pd.isnull(row[aspect]):
        pass
    else: 
        print("Passive ", i, ": ", 
              row[aspect], sep='')

In [69]:
# Left-alignment was easier this way...
for i, row in daemons.iterrows():
    print("**", row["name"], "**", sep='')
    print("```")
    print("Skill: ", row["skill"], sep='')
    print_if("ability 1", 1)
    print_if("ability 2", 2)
    print("```")

**Poli'ahu [Xmas]**
```
Skill: Can be used as bond and in guild conquest.
Passive 1: Absorbs a 15% of DMG dealt as HP.
```
**Natalia Nyfe [Xmas]**
```
Skill: Reduces all enemies' DEF by {delay1} for a limited time.
Passive 1: All allies take 3% less DMG.
```
**Kiku Ichimonji [Xmas]**
```
Skill: Increases the Crit Rate of 2 allies with the highest ATK by {delay1} for a limited time.
Passive 1: Increases 10% of Skill DMG and increases 60% of Crit DMG.
```
**Hashihime [Xmas]**
```
Skill: Deals {value} DMG to 3 enemies (ranged priority).
Passive 1: Increases 17% of skill DMG of the 2 allies with highest ATK.
```
**Snow Dome**
```
Skill: Can be used as bond and in guild conquest.
Passive 1: At the start of the final wave, enemy with the most HP takes 12% more damage.
```
**Santa Claus**
```
Skill: Deals {value} DMG to the 3 enemies with the highest ATK.
Passive 1: Increases 15% of skill DMG dealt at the start of each wave.
```
**Salamander [Xmas]**
```
Skill: Restores {value} HP. Target dea